# Inspect a saved DockingMT result from an installed package

Read a caller-supplied `DockingResult.to_dict()` JSON record using public APIs.
This example keeps the result detached and does not prepare, reconstruct, dock,
score, calculate RMSD or open a viewer. It requires the ordinary installed package
and its declared dependencies, with Jupyter tooling in the selected kernel.

Before starting the kernel, set `DOCKINGMT_RESULT_PATH` to your `.json` or
`.json.gz` file and `DOCKINGMT_RESULT_SHA256` to a digest obtained from a trusted
source. You can instead edit the two assignments in the first cell. No repository,
`devtools` module, private wheel path or qualification harness is required.

A matching external digest identifies the supplied bytes relative to that trust
choice. It does not establish chemistry, software authenticity or complete case
provenance. The internal record audit and captured-input checks below answer
separate questions. The original producer's recorded versions stay separate from
the installed reader's version. See the [companion guide](installed_result_inspection.md) for the public example
file, its externally recorded digest and the exact tested installation.

In [ ]:
import gzip
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import sys
from copy import deepcopy
from pathlib import Path

import pyunitwizard as puw
from smonitor.integrations import CatalogException

import dockingmt as dmt

result_path = Path(os.environ['DOCKINGMT_RESULT_PATH']).expanduser().resolve()
expected_file_sha256 = os.environ['DOCKINGMT_RESULT_SHA256'].lower()
if len(expected_file_sha256) != 64 or any(
    char not in '0123456789abcdef' for char in expected_file_sha256
):
    raise ValueError('Supply an externally expected SHA-256 digest.')

prefix = Path(sys.prefix).resolve()
reader_origin = Path(dmt.__file__).resolve()
distribution = metadata.distribution('dockingmt')
installation = json.loads(distribution.read_text('direct_url.json') or '{}')
if not reader_origin.is_relative_to(prefix) or installation.get('dir_info', {}).get(
    'editable', False
):
    raise RuntimeError(
        'Select an ordinary-installed DockingMT kernel without a source overlay.'
    )
assert dmt.__version__ == metadata.version('dockingmt')
print({'reader_version': dmt.__version__, 'python': platform.python_version()})

## Identify the file before interpreting it

The digest covers the actual file, including gzip encoding when used. JSON
loading is ordinary record IO, not molecular-system conversion. Unsupported
schema versions will be refused by DockingMT's existing reader. No sidecar or
newly calculated digest is automatically promoted to an external trust root.

In [ ]:
input_bytes = result_path.read_bytes()
actual_file_sha256 = hashlib.sha256(input_bytes).hexdigest()
if actual_file_sha256 != expected_file_sha256:
    raise ValueError(
        'The supplied file does not match the externally expected identity.'
    )
encoding = 'gzip' if result_path.suffix == '.gz' else 'identity'
decoded = gzip.decompress(input_bytes) if encoding == 'gzip' else input_bytes


def reject_nonfinite(value):
    raise ValueError('Nonfinite JSON constants are unsupported.')


record = json.loads(decoded.decode('utf-8'), parse_constant=reject_nonfinite)
original_record = deepcopy(record)
print(
    {
        'file_identity_matches': True,
        'encoding': encoding,
        'file_bytes': len(input_bytes),
        'decoded_bytes': len(decoded),
    }
)

## Audit evidence and read a detached result

`consistent` means the supported recorded claims agree. `incomplete` means some
checks lack evidence; it is not complete provenance. `inconsistent` means a
present claim is malformed or contradictory, so this example stops. The public
reader additionally admits the record schema.

Captured PDBQT checks validate bytes against recorded hashes. They do not parse
molecular syntax or restore missing chemistry. When captures are absent, their
status remains unavailable; no replacement bytes or declarations are invented.

In [ ]:
audit = dmt.audit_result(record)
if audit['status'] == 'inconsistent':
    raise ValueError(
        'The record contains inconsistent evidence; inspect audit["checks"].'
    )
result = dmt.DockingResult.from_dict(record)
assert result.problem is None
result_snapshot = result.to_dict()

provenance = record.get('provenance') or {}
capture_checks = [
    check for check in audit['checks'] if check['check'] == 'captured_input'
]
captures_available = len(capture_checks) == 2 and all(
    check['status'] == 'consistent' for check in capture_checks
)
captured_sizes = None
if captures_available:
    captured_sizes = dmt.verify_captured_inputs(provenance['backend_artifacts'])

print(
    {
        'record_audit': audit['status'],
        'poses': len(result.poses),
        'problem_attached': result.problem is not None,
        'captured_bytes_verified': captures_available,
        'captured_sizes_bytes': captured_sizes,
    }
)

## Inspect stored identity, order, scores and units

Collection index, one-based display position, optional scientific rank and pose
identity are distinct. Missing ranks/IDs stay missing; zero scores remain zero.
No sorting, rescoring or inference from a score name occurs. A descriptor is a
recorded declaration; missing units/methods remain unknown.

Coordinates below are simply stored values converted explicitly to angstrom.
The unit-roundtrip check is numerical record validation, not molecular analysis,
reference matching, RMSD, fitting or chemical certification.

In [ ]:
rows = []
for index, (pose, saved) in enumerate(
    zip(result.poses, record.get('poses', []), strict=True)
):
    assert pose.pose_id == saved.get('pose_id')
    assert pose.rank == saved.get('rank')
    assert pose.partner_state_id == saved.get('partner_state_id')
    assert pose.receptor_state_id == saved.get('receptor_state_id')
    assert pose.scores == saved.get('scores', {})
    definitions = pose.score_definitions
    assert definitions == (saved.get('metadata') or {}).get('score_definitions', {})
    coordinate_record = saved['coordinates']
    expected_coordinates = puw.quantity(
        coordinate_record['value'], coordinate_record['unit']
    )
    assert puw.are_close(pose.coordinates, expected_coordinates, rtol=1e-12, atol=0)
    score_rows = []
    for name, value in pose.scores.items():
        definition = definitions.get(name) or {}
        score_rows.append(
            {
                'name': name,
                'value': value,
                'unit': definition.get('unit'),
                'method': definition.get('method'),
                'component': definition.get('component'),
                'kind': definition.get('kind'),
            }
        )
    coordinates_angstrom = puw.get_value(
        pose.coordinates, to_unit='angstrom', value_type=list
    )
    rows.append(
        {
            'index': index,
            'position': index + 1,
            'rank': pose.rank,
            'pose_id': pose.pose_id,
            'n_atoms': pose.n_atoms,
            'scores': score_rows,
            'first_stored_coordinate': coordinates_angstrom[:1],
            'coordinate_unit': 'angstrom',
        }
    )
print(json.dumps(rows, indent=2, allow_nan=False))

## An informative refusal on detached copies

An unsupported schema must fail before interpretation. When original captures
are available, changing a copied partner digest must also produce both an
inconsistent audit and a captured-byte refusal. SMonitor's public exception base
and structured fields identify the diagnosed failure; arbitrary exceptions are
not accepted as success. The original record and completed result stay intact.

In [ ]:
wrong_schema = deepcopy(record)
wrong_schema['schema_version'] = 'unsupported-tutorial-control'
assert dmt.audit_result(wrong_schema)['status'] == 'inconsistent'
try:
    dmt.DockingResult.from_dict(wrong_schema)
except CatalogException as error:
    assert error.extra.get('arg_name') == 'schema_version'
    schema_refusal = {'code': error.code, 'argument': error.extra['arg_name']}
else:
    raise AssertionError('The unsupported schema was not refused.')

capture_refusal = None
if captures_available:
    damaged = deepcopy(record)
    digest = damaged['provenance']['backend_artifacts']['partner']['sha256']
    damaged['provenance']['backend_artifacts']['partner']['sha256'] = (
        '0' if digest[0] != '0' else '1'
    ) + digest[1:]
    damaged_audit = dmt.audit_result(damaged)
    assert any(
        check['check'] == 'captured_input'
        and check['path'] == '/provenance/backend_artifacts/partner'
        and check['status'] == 'inconsistent'
        for check in damaged_audit['checks']
    )
    try:
        dmt.verify_captured_inputs(damaged['provenance']['backend_artifacts'])
    except CatalogException as error:
        assert error.extra.get('arg_name') == 'artifacts'
        assert error.extra.get('role') == 'partner'
        capture_refusal = {
            'code': error.code,
            'argument': error.extra['arg_name'],
            'role': 'partner',
        }
    else:
        raise AssertionError('The changed captured-input digest was not refused.')
assert record == original_record
assert result.to_dict() == result_snapshot
print(
    {
        'unsupported_schema_refused': schema_refusal,
        'damaged_capture_refused': capture_refusal,
    }
)

## Repeat reading under a different unit policy

The reader preserves physical quantities and stored score declarations when the
application chooses picometres/femtoseconds/kJ per mole. Stored coordinates may
use the object's explicit internal unit; that is distinct from the application's
standard-unit preference. No score is silently converted or relabelled.

In [ ]:
policy_before = puw.configure.report()
with puw.context(
    standard_units=['pm', 'fs', 'K', 'mole', 'dalton', 'e', 'kJ/mol', 'degrees']
):
    chosen_policy = puw.configure.report()
    policy_result = dmt.DockingResult.from_dict(record)
    assert policy_result.problem is None
    assert policy_result.to_dict() == result_snapshot
    assert puw.configure.report() == chosen_policy
assert puw.configure.report() == policy_before
print(
    {'nondefault_policy_preserves_record_meaning': True, 'caller_policy_restored': True}
)

## Serialize an independent record without restoring a molecular system

`to_dict` returns an owned snapshot, and `from_dict` reads it independently.
This does not authenticate an external bundle or recreate its molecular inputs.
A later `reconstruct_problem` call requires the actual inputs and checks available
recorded evidence; it is intentionally not called here. General molecular
persistence belongs to MolSysMT.

In [ ]:
exported = result.to_dict()
reread = dmt.DockingResult.from_dict(exported)
assert reread.problem is None and reread.to_dict() == result_snapshot
exported['poses'].clear()
assert result.to_dict() == reread.to_dict() == result_snapshot
assert record == original_record
assert hashlib.sha256(result_path.read_bytes()).hexdigest() == actual_file_sha256
print(
    {
        'independent_roundtrip': True,
        'original_input_unchanged': True,
        'molecular_problem_restored': False,
    }
)

## Keep a compact receipt

The receipt names the file identity and the reader while retaining the producer's
recorded versions separately. Recorded producer versions are claims in the
supplied file, not independent verification of historical software bytes.
No input path, captured molecular payload or operational backend filename is
copied into this summary. The notebook output itself retains the receipt.

In [ ]:
receipt = {
    'schema': 'dockingmt.installed_result_inspection@1',
    'scope': 'saved_result_record_inspection',
    'input': {
        'file_sha256': actual_file_sha256,
        'decoded_sha256': hashlib.sha256(decoded).hexdigest(),
        'encoding': encoding,
        'external_expected_identity_matches': True,
    },
    'recorded_producer': {
        name: provenance.get(name)
        for name in (
            'dockingmt_version',
            'molsysmt_version',
            'backend',
            'backend_version',
        )
    },
    'reader': {
        'dockingmt_version': dmt.__version__,
        'python': platform.python_version(),
        'pyunitwizard_version': metadata.version('pyunitwizard'),
        'smonitor_version': metadata.version('smonitor'),
        'module_origin_within_prefix': str(reader_origin.relative_to(prefix)),
        'editable': False,
        'archive_sha256_if_recorded': installation.get('archive_info', {})
        .get('hashes', {})
        .get('sha256'),
    },
    'record_audit_status': audit['status'],
    'pose_count': len(result.poses),
    'captured_bytes_verified': captures_available,
    'captured_sizes_bytes': captured_sizes,
    'schema_refusal': schema_refusal,
    'capture_refusal': capture_refusal,
    'identity_order_scores_and_descriptors_preserved': True,
    'coordinate_roundtrip_relative_tolerance': 1e-12,
    'coordinate_roundtrip_absolute_tolerance': 0,
    'coordinate_display_unit': 'angstrom',
    'nondefault_policy_preserved': True,
    'caller_policy_restored': True,
    'detached_roundtrip_verified': True,
    'input_file_and_record_unchanged': True,
    'molecular_problem_restored': False,
    'whole_case_authentication': False,
    'historical_software_bytes_verified': False,
    'chemical_validity_verified': False,
    'browser_or_C6_acceptance': False,
}
print(json.dumps(receipt, indent=2, allow_nan=False))